Phase 0

Phase 1 : Séparer les données proprement, train / validation / test

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    if val_size == 0:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        X_val = X_test[:0]
        y_val = y_test[:0]
        print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
        return X_train, X_val, X_test, y_train, y_val, y_test

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, random_state=random_state, stratify=y_temp
    )

    print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
    t_train, t_val, t_test = np.mean(y_train), np.mean(y_val), np.mean(y_test)
    stratif_ok = max(t_train, t_val, t_test) - min(t_train, t_val, t_test) < 0.02
    print(f"Taux classe positive — train: {t_train:.3f} | val: {t_val:.3f} | test: {t_test:.3f}")
    print(f"Répartition des classes conservée dans chaque jeu : {'oui' if stratif_ok else 'non'}")
    return X_train, X_val, X_test, y_train, y_val, y_test

In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("CAS NORMAL")
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)
assert len(X_train) + len(X_val) + len(X_test) == len(X)
print("Somme des tailles = total : OK")

print("\nCAS LIMITE — val_size=0")
split_train_val_test(X, y, val_size=0)

print("\nCAS ADVERSARIAL — 95/5")
y_desq = pd.Series([0]*950 + [1]*50)
X_desq = pd.DataFrame(np.random.randn(1000, 5))
split_train_val_test(X_desq, y_desq)

CAS NORMAL
Train : 341 | Validation : 114 | Test : 114
Taux classe positive — train: 0.628 | val: 0.623 | test: 0.632
Répartition des classes conservée dans chaque jeu : oui
Somme des tailles = total : OK

CAS LIMITE — val_size=0
Train : 455 | Validation : 0 | Test : 114

CAS ADVERSARIAL — 95/5
Train : 600 | Validation : 200 | Test : 200
Taux classe positive — train: 0.050 | val: 0.050 | test: 0.050
Répartition des classes conservée dans chaque jeu : oui


(            0         1         2         3         4
 538 -0.383460  1.227269 -1.753568  0.596488  0.713821
 872 -0.336031 -0.750868  1.845452  0.708339 -1.892423
 51  -0.733399 -0.225162  1.973173  1.918835  0.726008
 69  -0.370908  0.864025  0.013533 -0.019592 -1.956204
 939  0.465717  2.044373  0.942834  1.646267  0.712654
 ..        ...       ...       ...       ...       ...
 932  0.133827  0.347089  0.856054  1.402177 -0.709862
 761  1.013360 -0.850364 -1.553449  1.259510  1.170030
 829  0.422422  1.164415 -0.868979 -1.591502  0.415304
 857  0.529652 -1.150564  1.061640 -0.112973 -0.006801
 175 -0.597611  0.825963 -0.990126 -0.264976  0.556677
 
 [600 rows x 5 columns],
             0         1         2         3         4
 36   0.608808 -0.582476  1.213295  0.070413 -2.122537
 601 -0.848091 -1.018570 -0.847605  0.695430 -0.661419
 694  0.990452  0.589922  0.212396  1.009656  0.557861
 786  2.251860  0.662874 -0.054703  2.034168  1.001594
 465  0.153944 -0.018487 -1.482901 -0.

Les résultat semble correct : 
569 en cas  normal 
cas limite  validation = 0 sans planter 
et 5% de class positive en cas adversariale

